In [1]:
import findspark
findspark.init()

import time
import pandas as pd
import numpy as np
import math
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, collect_list, lit, udf
from pyspark.sql.types import DoubleType

In [2]:
# ==============================================================================
# 1. INITIALIZE SPARK SESSION
# ==============================================================================
spark = SparkSession.builder \
    .appName("ExplicitALSFinal") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

print("Spark Session created.")

# <<< CRITICAL FIX: Set a directory for checkpointing. Spark needs this.
# Make sure this directory exists or create it.
checkpoint_dir = "/tmp/spark_checkpoints"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir) # Clean up previous runs
os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

print("Spark Session created and checkpoint directory set.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/17 19:21:57 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.222.8.252 instead (on interface en0)
25/11/17 19:21:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 19:21:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session created.
Spark Session created and checkpoint directory set.


In [3]:
user_item_ratings_subratings = pd.read_csv('../data/filtered/dense_user_item_ratings.csv')
ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
# ratings_df = ratings_df.sample(n=10_000_000, random_state=42)
# drop duplicates
print(f"Original shape: {ratings_df.shape}")
ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
print(f"Shape after dropping duplicates: {ratings_df.shape}")

# load data as half-precision floats and integers as half ints to save memory
print(f"User: {ratings_df['user_id'].nunique()}, Item: {ratings_df['item_id'].nunique()}")

# Convert pandas DataFrame to Spark DataFrame
ratings_spark_df = spark.createDataFrame(ratings_df)    
ratings_spark_df

Original shape: (1400102, 3)
Shape after dropping duplicates: (1400102, 3)
User: 46660, Item: 27197


DataFrame[user_id: bigint, item_id: bigint, rating: double]

In [4]:
# ==============================================================================
# 3. SPLIT DATA AND TRAIN THE ALS MODEL
# ==============================================================================
(training_df, test_df) = ratings_spark_df.randomSplit([0.8, 0.2], seed=42)
# <<< CRITICAL FIX: Checkpoint the DataFrames.
# This materializes the data and truncates the query plan, preventing StackOverflowError.
training_df = training_df.checkpoint()
test_df = test_df.checkpoint()
print("Training and test DataFrames have been checkpointed.")

# Instantiate the ALS model
als = ALS(
    maxIter=100,
    regParam=0.01,
    rank=100,
    userCol="user_id",
    itemCol="item_id",
    ratingCol="rating",
    coldStartStrategy="drop", # Drop users/items in test set not in training
    nonnegative=True,         # Ratings are non-negative
    implicitPrefs=False       # <<< THIS IS THE CRITICAL PARAMETER FOR EXPLICIT FEEDBACK
)

print("Training the powerful ALS model...")
start_time = time.time()
model = als.fit(training_df)
print(f"Model training completed in {time.time() - start_time:.2f} seconds.")



25/11/17 19:22:19 WARN TaskSetManager: Stage 0 contains a task of very large size (2151 KiB). The maximum recommended task size is 1000 KiB.
25/11/17 19:22:21 WARN TaskSetManager: Stage 1 contains a task of very large size (2151 KiB). The maximum recommended task size is 1000 KiB.


Training and test DataFrames have been checkpointed.
Training the powerful ALS model...


25/11/17 19:22:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Model training completed in 226.70 seconds.


In [5]:
# ==============================================================================
# 4. EVALUATE RATING PREDICTION (MAE & RMSE)
# ==============================================================================
predictions = model.transform(test_df)
predictions.cache() # Cache predictions for reuse
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) on test data = {rmse:.4f}")
print(f"Mean Absolute Error (MAE) on test data = {mae:.4f}")

Root Mean Squared Error (RMSE) on test data = 0.8660
Mean Absolute Error (MAE) on test data = 0.6862


In [6]:
# ==============================================================================
# 5. EVALUATE RANKING METRICS (Precision@k, Recall@k, nDCG@k)
# ==============================================================================
print("Evaluating ranking metrics...")
K_VALUES = [1, 3, 5, 7, 10]
relevance_threshold = 4.0

all_user_recs = model.recommendForAllUsers(max(K_VALUES))
all_user_recs = all_user_recs.withColumn("recommendations", col("recommendations.item_id"))

ground_truth = test_df.where(col("rating") >= relevance_threshold) \
    .groupBy("user_id") \
    .agg(collect_list("item_id").alias("relevant_items"))

eval_df = all_user_recs.join(ground_truth, "user_id", "inner")
eval_df.cache()

# Define metric calculation functions
def precision_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return len(set(rec_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return len(set(rec_k) & set(relevant)) / len(relevant) if relevant else 0.0

def ndcg_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    # <<< FIX: Use math.log2 instead of np.log2 to avoid PickleException
    dcg = sum(1.0 / math.log2(i + 2) for i, item in enumerate(rec_k) if item in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

# Create UDFs for each metric
precision_udf = udf(precision_at_k, DoubleType())
recall_udf = udf(recall_at_k, DoubleType())
ndcg_udf = udf(ndcg_at_k, DoubleType())

# Calculate all metrics
ranking_metrics = {}
for k in K_VALUES:
    metrics_df = eval_df.select(
        precision_udf(col("recommendations"), col("relevant_items"), lit(k)).alias("precision"),
        recall_udf(col("recommendations"), col("relevant_items"), lit(k)).alias("recall"),
        ndcg_udf(col("recommendations"), col("relevant_items"), lit(k)).alias("ndcg")
    )
    
    agg_metrics = metrics_df.agg(
        {"precision": "avg", "recall": "avg", "ndcg": "avg"}
    ).first()
    
    ranking_metrics[f'Precision@{k}'] = agg_metrics['avg(precision)']
    ranking_metrics[f'Recall@{k}'] = agg_metrics['avg(recall)']
    ranking_metrics[f'nDCG@{k}'] = agg_metrics['avg(ndcg)']

Evaluating ranking metrics...


In [7]:
# ==============================================================================
# 6. PRINT FINAL METRICS
# ==============================================================================
print("\n--- FINAL EVALUATION RESULTS ---")
print("\n--- Rating Prediction Metrics ---")
print(f"Root Mean Squared Error (RMSE) = {rmse:.4f}")
print(f"Mean Absolute Error (MAE)      = {mae:.4f}")

print("\n--- Ranking Metrics ---")
for k in K_VALUES:
    print(f"Precision@{k:<2} = {ranking_metrics.get(f'Precision@{k}', 0):.4f}")
    print(f"Recall@{k:<5} = {ranking_metrics.get(f'Recall@{k}', 0):.4f}")
    print(f"nDCG@{k:<6} = {ranking_metrics.get(f'nDCG@{k}', 0):.4f}")
    print("-" * 30)


--- FINAL EVALUATION RESULTS ---

--- Rating Prediction Metrics ---
Root Mean Squared Error (RMSE) = 0.8660
Mean Absolute Error (MAE)      = 0.6862

--- Ranking Metrics ---
Precision@1  = 0.0004
Recall@1     = 0.0001
nDCG@1      = 0.0004
------------------------------
Precision@3  = 0.0005
Recall@3     = 0.0003
nDCG@3      = 0.0005
------------------------------
Precision@5  = 0.0004
Recall@5     = 0.0005
nDCG@5      = 0.0005
------------------------------
Precision@7  = 0.0004
Recall@7     = 0.0006
nDCG@7      = 0.0006
------------------------------
Precision@10 = 0.0004
Recall@10    = 0.0009
nDCG@10     = 0.0007
------------------------------


In [5]:
# ==============================================================================
# 6. CLEAN UP
# ==============================================================================
spark.stop()
# Clean up the checkpoint directory
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
print("\nSpark session stopped and checkpoint directory cleaned up.")

ConnectionRefusedError: [Errno 61] Connection refused

# implit lib

In [1]:
import pandas as pd
import numpy as np
import time
from scipy.sparse import csr_matrix, lil_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.model_selection import train_test_split

In [2]:
user_item_ratings_subratings = pd.read_csv('../data/filtered//dense_user_item_ratings.csv')
ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
# ratings_df = ratings_df.sample(n=10_000_000, random_state=42)
# drop duplicates
print(f"Original shape: {ratings_df.shape}")
# ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
print(f"Shape after dropping duplicates: {ratings_df.shape}")

# load data as half-precision floats and integers as half ints to save memory
print(f"User: {ratings_df['user_id'].nunique()}, Item: {ratings_df['item_id'].nunique()}")
ratings_df.info()

Original shape: (1400102, 3)
Shape after dropping duplicates: (1400102, 3)
User: 46660, Item: 27197
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400102 entries, 0 to 1400101
Data columns (total 3 columns):
 #   Column   Non-Null Count    Dtype  
---  ------   --------------    -----  
 0   user_id  1400102 non-null  int64  
 1   item_id  1400102 non-null  int64  
 2   rating   1400102 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 32.0 MB


In [3]:
# ==============================================================================
# 2. EFFICIENT TRAIN-TEST SPLIT (User-Stratified)
# ==============================================================================
def user_stratified_split(df, test_size=0.05):
    """
    Performs a user-stratified split. For each user, a portion of their ratings
    is moved to the test set. Users with only one rating are kept entirely in the train set.
    """
    print(f"Performing user-stratified split with test_size={test_size}...")
    train_list = []
    test_list = []
    
    # Group by user and iterate
    for user_id, group in df.groupby('user_id'):
        if len(group) > 1:
            # If user has more than one rating, split them
            train_ratings, test_ratings = train_test_split(group, test_size=test_size, random_state=42)
            train_list.append(train_ratings)
            test_list.append(test_ratings)
        else:
            # If user has only one rating, it must go to the training set
            train_list.append(group)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)
    
    print(f"Train set size: {len(train_df)}, Test set size: {len(test_df)}")
    return train_df, test_df

In [4]:
# ==============================================================================
# 3. MAPPING AND SPARSE MATRIX CREATION
# ==============================================================================
def create_sparse_matrix(df, user_map, item_map, num_users, num_items):
    """Creates a scipy sparse CSR matrix from a ratings DataFrame."""
    print("Creating sparse user-item matrix...")
    # Use LIL (List of Lists) format for efficient incremental construction
    mat = lil_matrix((num_users, num_items))
    # Map user and item IDs to matrix indices
    user_indices = df['user_id'].map(user_map)
    item_indices = df['item_id'].map(item_map)
    for user_idx, item_idx, rating in zip(user_indices, item_indices, df['rating']):
        mat[user_idx, item_idx] = rating
    # Convert to CSR for fast matrix-vector operations
    return mat.tocsr()

In [5]:
# # 1. Create unique integer mappings for users and items
# # This is crucial for creating the matrix indices
# user_ids = ratings_df['user_id'].unique()
# item_ids = ratings_df['item_id'].unique()

# user_id_map = {id: i for i, id in enumerate(user_ids)}
# item_id_map = {id: i for i, id in enumerate(item_ids)}

# # Map the original IDs to the new integer indices
# ratings_df['user_idx'] = ratings_df['user_id'].map(user_id_map)
# ratings_df['item_idx'] = ratings_df['item_id'].map(item_id_map)

# # 2. Create the sparse matrix
# # Use Compressed Sparse Row (CSR) format for efficient row slicing
# num_users = len(user_ids)
# num_items = len(item_ids)

# user_item_matrix = csr_matrix((ratings_df['rating'],
#                                (ratings_df['user_idx'], ratings_df['item_idx'])),
#                               shape=(num_users, num_items))

# print(f"Shape of the sparse matrix: {user_item_matrix.shape}")
# print(f"Sparsity: {100 * (1 - (user_item_matrix.nnz / (num_users * num_items))):.2f}%")

In [6]:
K_VALUES = [1, 3, 5, 7, 10]
TEST_SET_SIZE = 0.05

In [7]:
# ===================== FIX 3: GUARANTEED CONSISTENCY PIPELINE =====================
# STEP 1: Create a single, global mapping from the FULL dataset BEFORE splitting.
# This is the single source of truth for all user and item indices.
print("Creating global user and item mappings...")
all_user_ids = sorted(ratings_df['user_id'].unique())
all_item_ids = sorted(ratings_df['item_id'].unique())
user_map = {id: i for i, id in enumerate(all_user_ids)}
item_map = {id: i for i, id in enumerate(all_item_ids)}
num_total_users = len(user_map)
num_total_items = len(item_map)
print(f"Total unique users: {num_total_users}, Total unique items: {num_total_items}")

# STEP 2: Split the data.
train_df, test_df = user_stratified_split(ratings_df, test_size=TEST_SET_SIZE)

# STEP 3: Create the training matrix using the global mappings.
item_user_train_matrix = create_sparse_matrix(train_df, user_map, item_map, num_total_users, num_total_items)
print(f"User-Item training matrix shape: {item_user_train_matrix.shape}")

Creating global user and item mappings...
Total unique users: 46660, Total unique items: 27197
Performing user-stratified split with test_size=0.05...
Train set size: 1303276, Test set size: 96826
Creating sparse user-item matrix...
User-Item training matrix shape: (46660, 27197)


In [8]:
# 4. Train the Model
print("\nTraining ALS model...")
start_time = time.time()
# Initialize the model
# Note: Implicit library expects items x users matrix
model = AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=50,
    calculate_training_loss=True,
    use_gpu=False # Set to True if you have a CUDA-enabled GPU
)

# Train the model
user_item_train_matrix = item_user_train_matrix.T.tocsr()
print(f"Item-user training matrix shape: {item_user_train_matrix.shape}")
model.fit(item_user_train_matrix)
end_time = time.time()
print(f"Model training completed in {end_time - start_time:.2f} seconds.")


Training ALS model...
Item-user training matrix shape: (46660, 27197)


/opt/anaconda3/lib/python3.11/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

Model training completed in 101.70 seconds.


In [9]:
# Save the model
import pickle
with open('models/als_hotelrec_model_v5.pkl', 'wb') as f:
    pickle.dump((model, user_map, item_map), f)

In [10]:
# ==============================================================================
# 4. METRIC CALCULATION FUNCTIONS
# ==============================================================================
def precision_at_k(recommended_items, relevant_items, k):
    rec_k = recommended_items[:k]
    return len(set(rec_k) & set(relevant_items)) / k

def recall_at_k(recommended_items, relevant_items, k):
    rec_k = recommended_items[:k]
    return len(set(rec_k) & set(relevant_items)) / len(relevant_items)

def ndcg_at_k(recommended_items, relevant_items, k):
    rec_k = recommended_items[:k]
    dcg = 0.0
    for i, item in enumerate(rec_k):
        if item in relevant_items:
            dcg += 1.0 / np.log2(i + 2) # log base 2 of position + 1 (i+2 since i is 0-indexed)
    
    # Ideal DCG
    idcg = 0.0
    num_relevant = min(len(relevant_items), k)
    for i in range(num_relevant):
        idcg += 1.0 / np.log2(i + 2)
        
    return dcg / idcg if idcg > 0 else 0.0

# ==============================================================================
# 5. HYPER-EFFICIENT BATCHED EVALUATION (CORRECTED AND ROBUST)
# ==============================================================================
def efficient_evaluation(model, train_matrix, test_df, user_map, item_map, k_values, batch_size=1000, relevance_threshold=4.0):
    print(f"Starting hyper-efficient evaluation with batch_size={batch_size}...")
    
    # ============================= FIX 1: CORRECT REVERSE MAPPING =============================
    # Correctly create the reverse mapping from an index back to the original item ID
    idx_to_item = {idx: original_id for original_id, idx in item_map.items()}
    # ==========================================================================================

    metrics = {'mae': 0.0, 'rmse': 0.0, 'total_error_ratings': 0,
               'precision': {k: 0.0 for k in k_values}, 'recall': {k: 0.0 for k in k_values}, 'ndcg': {k: 0.0 for k in k_values}}
    
    test_user_groups = test_df.groupby('user_id')
    test_user_ids = list(test_user_groups.groups.keys())
    num_test_users = len(test_user_ids)
    
    max_k = max(k_values)
    
    model_num_users = model.user_factors.shape[0]
    model_num_items = model.item_factors.shape[0]
    print(f"Model was trained on {model_num_users} users and {model_num_items} items.")
    
    for i in range(0, num_test_users, batch_size):
        batch_user_ids_original = test_user_ids[i:i+batch_size]
        batch_user_indices_original = [user_map[uid] for uid in batch_user_ids_original]

        valid_batch_pairs = [(uid, uidx) for uid, uidx in zip(batch_user_ids_original, batch_user_indices_original) if uidx < model_num_users]
        if not valid_batch_pairs: continue
        batch_user_ids, batch_user_indices = zip(*valid_batch_pairs)

        user_items_batch = train_matrix[list(batch_user_indices)]
        recommendations_batch = model.recommend(userid=list(batch_user_indices), user_items=user_items_batch, N=max_k, filter_already_liked_items=True)

        for j, user_id in enumerate(batch_user_ids):
            user_idx = batch_user_indices[j]
            user_group = test_user_groups.get_group(user_id)
            
            relevant_items = set(user_group[user_group['rating'] >= relevance_threshold]['item_id'])
            if not relevant_items: continue
                
            recommended_indices = recommendations_batch[0][j]
            recommended_items = [idx_to_item[idx] for idx in recommended_indices if idx in idx_to_item]

            for k in k_values:
                metrics['precision'][k] += precision_at_k(recommended_items, relevant_items, k)
                metrics['recall'][k] += recall_at_k(recommended_items, relevant_items, k)
                metrics['ndcg'][k] += ndcg_at_k(recommended_items, relevant_items, k)
                
            # =================== FIX 2: ROBUST INDEXING FOR ERROR METRICS ===================
            # Ensure all items from test_df exist in the global item_map before processing
            test_items_for_user = [(row.item_id, row.rating) for row in user_group.itertuples() if row.item_id in item_map]
            if not test_items_for_user: continue
            
            test_item_ids, true_ratings = zip(*test_items_for_user)
            test_item_indices = [item_map[iid] for iid in test_item_ids]
            # ================================================================================

            user_factors = model.user_factors[user_idx]
            item_factors = model.item_factors[test_item_indices]
            predicted_ratings = user_factors.dot(item_factors.T)
            
            for pred, true in zip(predicted_ratings, true_ratings):
                error = pred - true
                metrics['mae'] += abs(error)
                metrics['rmse'] += error**2
                metrics['total_error_ratings'] += 1

    final_metrics = {}
    if metrics['total_error_ratings'] > 0:
        final_metrics['MAE'] = metrics['mae'] / metrics['total_error_ratings']
        final_metrics['RMSE'] = np.sqrt(metrics['rmse'] / metrics['total_error_ratings'])

    for k in k_values:
        final_metrics[f'Precision@{k}'] = metrics['precision'][k] / num_test_users
        final_metrics[f'Recall@{k}'] = metrics['recall'][k] / num_test_users
        final_metrics[f'nDCG@{k}'] = metrics['ndcg'][k] / num_test_users

    return final_metrics

## new 1.4 mil data

In [12]:
# 5. Run Evaluation - 100E / 100F / 0.2testSet  / 0.005 reg / v4 50+
print("\nStarting evaluation...")
start_time = time.time()
final_results = efficient_evaluation(
    model=model,
    train_matrix=item_user_train_matrix,
    test_df=test_df,
    user_map=user_map,
    item_map=item_map,
    k_values=K_VALUES,
    batch_size=1024
)
end_time = time.time()
print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

# 6. Display Results
print("\n--- Evaluation Results ---")
for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")
print("--------------------------\n")


Starting evaluation...
Starting hyper-efficient evaluation with batch_size=1024...
Model was trained on 46660 users and 27197 items.
Evaluation completed in 16.10 seconds.

--- Evaluation Results ---
MAE: 4.0980
RMSE: 4.1832
Precision@1: 0.0089
Recall@1: 0.0051
nDCG@1: 0.0089
Precision@3: 0.0070
Recall@3: 0.0119
nDCG@3: 0.0106
Precision@5: 0.0063
Recall@5: 0.0180
nDCG@5: 0.0134
Precision@7: 0.0058
Recall@7: 0.0228
nDCG@7: 0.0153
Precision@10: 0.0052
Recall@10: 0.0292
nDCG@10: 0.0176
--------------------------



## for old 1.4 mil data

In [37]:
# 5. Run Evaluation - 100E / 100F / 0.2testSet  / 0.005 reg / v4 50+
print("\nStarting evaluation...")
start_time = time.time()
final_results = efficient_evaluation(
    model=model,
    train_matrix=user_item_train_matrix,
    test_df=test_df,
    user_map=user_map,
    item_map=item_map,
    k_values=K_VALUES,
    batch_size=1024
)
end_time = time.time()
print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

# 6. Display Results
print("\n--- Evaluation Results ---")
for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")
print("--------------------------\n")


Starting evaluation...
Starting hyper-efficient evaluation with batch_size=1024...
Model was trained on 19745 users and 146203 items.
Evaluation completed in 10.70 seconds.

--- Evaluation Results ---
MAE: 3.9544
RMSE: 4.0629
Precision@1: 0.0134
Recall@1: 0.0015
nDCG@1: 0.0134
Precision@3: 0.0141
Recall@3: 0.0049
nDCG@3: 0.0140
Precision@5: 0.0132
Recall@5: 0.0076
nDCG@5: 0.0135
Precision@7: 0.0126
Recall@7: 0.0102
nDCG@7: 0.0134
Precision@10: 0.0121
Recall@10: 0.0140
nDCG@10: 0.0143
--------------------------



In [10]:
# 5. Run Evaluation - 20E / 100F / 0.2testSet  / 0.005 reg / v3
print("\nStarting evaluation...")
start_time = time.time()
final_results = efficient_evaluation(
    model=model,
    train_matrix=train_matrix,
    test_df=test_df,
    user_map=user_map,
    item_map=item_map,
    k_values=K_VALUES,
    batch_size=1024
)
end_time = time.time()
print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

# 6. Display Results
print("\n--- Evaluation Results ---")
for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")
print("--------------------------\n")


Starting evaluation...
Starting hyper-efficient evaluation with batch_size=1024...
Model was trained on 250985 users.
Evaluation completed in 270.05 seconds.

--- Evaluation Results ---
MAE: 4.0754
RMSE: 4.1848
Precision@1: 0.0001
Recall@1: 0.0000
nDCG@1: 0.0001
Precision@3: 0.0001
Recall@3: 0.0001
nDCG@3: 0.0001
Precision@5: 0.0000
Recall@5: 0.0001
nDCG@5: 0.0001
Precision@7: 0.0000
Recall@7: 0.0001
nDCG@7: 0.0001
Precision@10: 0.0000
Recall@10: 0.0001
nDCG@10: 0.0001
--------------------------



In [11]:
# 5. Run Evaluation - 20E / 50F / 0.05testSet  / 0.005 reg / v2
print("\nStarting evaluation...")
start_time = time.time()
final_results = efficient_evaluation(
    model=model,
    train_matrix=train_matrix,
    test_df=test_df,
    user_map=user_map,
    item_map=item_map,
    k_values=K_VALUES,
    batch_size=1024
)
end_time = time.time()
print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

# 6. Display Results
print("\n--- Evaluation Results ---")
for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")
print("--------------------------\n")


Starting evaluation...
Starting hyper-efficient evaluation with batch_size=1024...
Model was trained on 250985 users.
Evaluation completed in 248.37 seconds.

--- Evaluation Results ---
MAE: 4.2795
RMSE: 4.3535
Precision@1: 0.0000
Recall@1: 0.0000
nDCG@1: 0.0000
Precision@3: 0.0000
Recall@3: 0.0001
nDCG@3: 0.0001
Precision@5: 0.0000
Recall@5: 0.0001
nDCG@5: 0.0001
Precision@10: 0.0000
Recall@10: 0.0001
nDCG@10: 0.0001
--------------------------

